In [1]:
import numpy as np
from scipy.sparse import csr_matrix
from qiskit import generate_preset_pass_manager, QuantumCircuit
from qiskit_aer import AerSimulator

from hamiltonian.base import HamiltonianType
from hamiltonian.free_wilson import FreeWilson2D

def unit_vector(vector):
    """ Returns the unit vector of the vector.  """
    return vector / np.linalg.norm(vector)

def angle_between(v1, v2):
    """ Returns the angle in radians between vectors 'v1' and 'v2'::

            >>> angle_between((1, 0, 0), (0, 1, 0))
            1.5707963267948966
            >>> angle_between((1, 0, 0), (1, 0, 0))
            0.0
            >>> angle_between((1, 0, 0), (-1, 0, 0))
            3.141592653589793
    """
    v1_u = unit_vector(v1)
    v2_u = unit_vector(v2)
    return np.arccos(np.clip(np.dot(v1_u, v2_u), -1.0, 1.0))

In [2]:
def calc_angles(circuit: QuantumCircuit, mass=-6):
    sv_name = "final"
    circuit.save_statevector(sv_name)

    hamiltonian = FreeWilson2D(2,2, mass, 1)
    # print(f"Using {hamiltonian}", flush=True)
    h_operator = hamiltonian.hamiltonian_op(HamiltonianType.Full)
    # noinspection PyTypeChecker
    h_sparse_matrix = h_operator.to_matrix(sparse=True)
    h_sparse_matrix: csr_matrix
    h_sparse_matrix.eliminate_zeros()


    simulator_options = {
        "method": "statevector",
    }
    backend = AerSimulator(**simulator_options)
    pm = generate_preset_pass_manager(backend=backend)
    circuit = pm.run(circuit)
    result = backend.run(circuit, shots=1).result()
    sv = result.data(0)[sv_name].data
    # print(f"SV: {sv}", flush=True)
    svp = h_sparse_matrix@sv
    # print(f"SVP: {svp}", flush=True)
    print(angle_between(sv,sv), angle_between(svp,svp), angle_between(sv,svp),flush=True)

# circuit, num_state_vectors = ansatz.build_full_ansatz_with_save_points()
num_qubits = 8

# Full superposition
circuit = QuantumCircuit(num_qubits)
for i in range(0, num_qubits):
    circuit.h(i)
print("\nFull superposition")
calc_angles(circuit)

# 10101010
circuit = QuantumCircuit(num_qubits)
for i in range(0, num_qubits, 2):
    circuit.x(i)
print("\n10101010")
calc_angles(circuit)

# Full superposition + SGates
circuit = QuantumCircuit(num_qubits)
for i in range(0, num_qubits):
    circuit.h(i)
    circuit.s(i)
print("\nFull superposition + SGates")
calc_angles(circuit)

# Empty
circuit = QuantumCircuit(num_qubits)
print("\nEmpty")
calc_angles(circuit)

# 00001111+11110000
circuit = QuantumCircuit(num_qubits)
circuit.h(0)
for i in range(num_qubits // 2, num_qubits):
    circuit.x(i)
for i in range(1, num_qubits):
    circuit.cx(0, i)
print("\n00001111+11110000")
calc_angles(circuit)

# 11001100+00110011
circuit = QuantumCircuit(num_qubits)
circuit.h(0)
for i in range(1, num_qubits):
    circuit.cx(0, i)
for i in range(0, num_qubits, 4):
    circuit.x(i)
for i in range(1, num_qubits, 4):
    circuit.x(i)
print("\n11001100+00110011")
calc_angles(circuit)

# (00+11)^4
circuit = QuantumCircuit(num_qubits)
for i in range(0, num_qubits, 2):
    circuit.h(i)
    circuit.cx(i, i + 1)
print("\n(00+11)^4")
calc_angles(circuit)


Full superposition
-0j (2.1073424255447014e-08-0j) (1.4966033467686606-0j)

10101010
-0j -0j (1.5707963267948966-0j)

Full superposition + SGates
(1.5707963267948966-8.128604488001096e-33j) (1.559807094624704+1.615016372328831e-17j) (1.4966033467686606+3.641091188106227e-17j)

Empty
-0j (nan+nanj) (nan+nanj)

00001111+11110000
-0j -0j (1.5707963267948966-0j)


/tmp/ipykernel_38008/2802809292.py:11: RuntimeWarning: invalid value encountered in divide
  return vector / np.linalg.norm(vector)



11001100+00110011
-0j (1.4901161193847656e-08-0j) (1.5707963267948966-0j)

(00+11)^4
-0j -0j (1.5707963267948966-0j)
